In [1]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do .env
load_dotenv()

# Caminho para o modelo salvo
training_model = os.getenv("TRAINING_MODEL")

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification

# Carrega tokenizer e modelo treinado
tokenizer = BertTokenizer.from_pretrained(training_model)
model = BertForSequenceClassification.from_pretrained(training_model)
# Coloca o modelo em modo de inferência
model.eval()  

d:\Visual Code\SentimentAnalysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [3]:
import torch

def predict_sentiment(text):
    # Tokeniza a entrada
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    # Inferência sem calcular gradiente
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][predicted_class].item()

    # Ajuste conforme seu mapeamento de rótulo
    label = "Positivo" if predicted_class == 1 else "Negativo"
    return {
        "label": label,
        "confidence": round(confidence * 100, 2)
    }


In [4]:
print(predict_sentiment("This movie was absolutely fantastic!"))
print(predict_sentiment("I hated the entire experience."))
print(predict_sentiment("It was okay, not great but not bad either."))

d:\Visual Code\SentimentAnalysis\venv\Lib\site-packages\torch\nn\modules\module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'label': 'Positivo', 'confidence': 98.96}
{'label': 'Negativo', 'confidence': 96.84}
{'label': 'Negativo', 'confidence': 63.49}
